# AURA Shield — DPO config-5 unseen-30 overfit check

**Before running:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`.

What this notebook does:
1. trains the config-5 LoRA DPO (Qwen2.5-1.5B, r16, beta=0.1, lr=2e-4, 10 epochs) **only if** no adapter is found;
2. scores base vs DPO on dev-29 (seen) and unseen-30 (never trained on);
3. prints one JSON block + downloads it as `dpo_unseen_qwen15.json`.

Runtime: ~40-50 min on a fresh T4; ~5 min if the adapter already exists.
The data comes from the public repo (committed files), so nothing needs uploading.

In [ ]:
# 1) Dependencies (~1 min). If a later cell fails on import after this, do
#    Runtime -> Restart session, then Run all again (install is cached).
!pip -q install -U "transformers>=4.45" "trl>=0.12" peft accelerate datasets

# Colab ships torchao 0.10, which current peft rejects with an ImportError when it
# builds LoRA layers (torchao is optional). Removing it makes peft skip that path.
!pip -q uninstall -y torchao


In [ ]:
# 2) Config — edit ADAPTER_DIR only if you saved the config-5 adapter elsewhere (e.g. Drive).
from pathlib import Path

ADAPTER_DIR = Path("/content/dpo_out")
TRAIN_IF_MISSING = True          # False = fail fast instead of retraining
BASE_ID = "Qwen/Qwen2.5-1.5B"
MAX_LEN = 512
RAW = "https://raw.githubusercontent.com/manaal6/Aura-Shield/main/data/"

print("adapter present:", (ADAPTER_DIR / "adapter_config.json").exists())

In [ ]:
# 3) Data + scoring helpers
import json
import torch
import urllib.request
from transformers import AutoTokenizer

def fetch(name):
    with urllib.request.urlopen(RAW + name) as r:
        return [json.loads(ln) for ln in r.read().decode().splitlines() if ln.strip()]

rows = fetch("dpo_preferences.jsonl")
train = [r for r in rows if r["split"] == "train"]
dev = [r for r in rows if r["split"] == "dev"]
unseen = fetch("dpo_generalization.jsonl")
assert (len(train), len(dev), len(unseen)) == (133, 29, 30), (len(train), len(dev), len(unseen))
print(f"train {len(train)} | dev {len(dev)} | unseen {len(unseen)}")

assert torch.cuda.is_available(), "No GPU — Runtime -> Change runtime type -> T4 GPU"
print("gpu:", torch.cuda.get_device_name(0))

tok = AutoTokenizer.from_pretrained(BASE_ID)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

@torch.no_grad()
def seq_logprob(model, prompt, resp):
    ids = tok(prompt + resp, return_tensors="pt", truncation=True,
              max_length=MAX_LEN).input_ids.to(model.device)
    logits = model(ids).logits[:, :-1].float()
    labels = ids[:, 1:]
    lp = torch.log_softmax(logits, -1).gather(-1, labels.unsqueeze(-1)).squeeze(-1)
    n_prompt = tok(prompt, return_tensors="pt").input_ids.shape[1] - 1
    return float(lp[:, n_prompt:].sum())

def evaluate(model, items):
    ok, margins = 0, []
    for r in items:
        m = seq_logprob(model, r["prompt"], r["chosen"]) - seq_logprob(model, r["prompt"], r["rejected"])
        margins.append(m)
        ok += m > 0
    n = len(items)
    return {"n": n, "preference_accuracy": f"{ok}/{n}={ok / n:.4f}",
            "mean_chosen_margin": round(sum(margins) / n, 2)}

print("helpers ready")

In [ ]:
# 4) Train config-5 DPO (skipped when an adapter is already present)
import time

if not (ADAPTER_DIR / "adapter_config.json").exists():
    assert TRAIN_IF_MISSING, f"no adapter at {ADAPTER_DIR} and TRAIN_IF_MISSING=False"
    from datasets import Dataset
    from peft import LoraConfig
    from transformers import AutoModelForCausalLM
    from trl import DPOConfig, DPOTrainer

    ds = Dataset.from_list([{"prompt": r["prompt"], "chosen": r["chosen"], "rejected": r["rejected"]}
                            for r in train])
    model = AutoModelForCausalLM.from_pretrained(BASE_ID, torch_dtype=torch.float16, device_map="auto")
    lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
                      target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
                      task_type="CAUSAL_LM")
    args = DPOConfig(output_dir=str(ADAPTER_DIR), beta=0.1, learning_rate=2e-4,
                     num_train_epochs=10, per_device_train_batch_size=4, fp16=True,
                     seed=7, max_length=MAX_LEN, logging_steps=10,
                     save_strategy="no", report_to=[])
    kw = dict(model=model, ref_model=None, args=args, train_dataset=ds, peft_config=lora)
    try:
        trainer = DPOTrainer(**kw, processing_class=tok)
    except TypeError:  # older trl
        trainer = DPOTrainer(**kw, tokenizer=tok)
    t0 = time.time()
    trainer.train()
    trainer.save_model(str(ADAPTER_DIR))
    print(f"trained in {time.time() - t0:.0f}s -> {ADAPTER_DIR}")
    del model, trainer
    torch.cuda.empty_cache()
else:
    print("adapter found — skipping training")

In [ ]:
# 5) Score base vs DPO (dev + unseen + train) and emit the record
from peft import PeftModel
from transformers import AutoModelForCausalLM

base = AutoModelForCausalLM.from_pretrained(BASE_ID, torch_dtype=torch.float16,
                                            device_map="auto").eval()
res = {"base_dev": evaluate(base, dev), "base_unseen": evaluate(base, unseen)}
dpo = PeftModel.from_pretrained(base, str(ADAPTER_DIR)).eval()
res |= {"dpo_dev": evaluate(dpo, dev),
        "dpo_unseen": evaluate(dpo, unseen),
        "dpo_train": evaluate(dpo, train)}

record = {
    "model": BASE_ID,
    "adapter": "LoRA r16 alpha32 q/k/v/o (config 5)",
    "adapter_dir": str(ADAPTER_DIR),
    "device": "cuda (Colab T4)",
    "beta": 0.1, "lr": 2e-4, "epochs": 10, "seed": 7,
    "n_train": 133, "n_dev": 29, "n_unseen": 30,
    **res,
    "note": ("generalization = dpo_unseen vs base_unseen; dpo_train checks overfit. "
             "Unseen was never trained on and is eval-only."),
}
print("=" * 72)
print("PASTE THIS JSON BACK INTO THE REPO")
print("=" * 72)
print(json.dumps(record, indent=2))

out = Path("/content/dpo_unseen_qwen15.json")
out.write_text(json.dumps(record, indent=2), encoding="utf-8")
try:
    from google.colab import files
    files.download(str(out))
except Exception as exc:  # noqa: BLE001
    print("(download skipped:", exc, ")")

## How to read the output

- `dpo_unseen` vs `base_unseen` is the actual claim: if DPO does not beat base on unseen phrasings, the dev win is memorization, not preference learning.
- `dpo_dev` should be around 22/29 if the retrain reproduced config 5 (small drift is expected across runs).
- `dpo_train` above ~0.9 with a flat unseen score is the overfit signature — say so, do not hide it.

Paste the JSON block back (or upload `dpo_unseen_qwen15.json`). It becomes
`results/kaust_three_pillars/dpo_lm/dpo_unseen_qwen15.json` and the DPO claim is
upgraded (or downgraded) honestly.